# A3T-GCN ablation (E0 / E1 / E2) on Kaggle

Runs `main.py` of this repository on a Kaggle GPU. **All final E0 / E1 / E2 runs (and the Stage B
selection runs) must use this same notebook and accelerator**, so that only the graph differs.

**Before running**
1. *Add data* → your private dataset with the LargeST-GBA files
   (`gba_his_raw_2021.h5`, `gba_rn_adj.npy`, `gba_meta.csv`).
2. *Add data* → your private dataset with the ablation artifacts
   (`E1__*.npz`, `E2__*.npz`, `sensor_filter_*.json`).
3. Settings: **Accelerator = GPU P100**, **Internet = On**.
4. Edit the *Run settings* cell, then **Save Version → Save & Run All (Commit)**.
   The run continues with the browser closed (max 12 h); results appear under *Output*.

## Run settings (edit only this cell)

In [ ]:
EXPERIMENT     = "E0"      # "E0", "E1" or "E2"
GRAPH_FILE     = None      # e.g. "E1__lag3_alpha1e+06.npz" (None for E0)
EPOCHS         = None      # None = config default (20); 3 for Stage B selection runs
SELECTION_ONLY = False     # True = train + validate only, test set NOT evaluated
RUN_NAME       = None      # output folder name; default = EXPERIMENT

REPO_URL       = "https://github.com/prof-sanjay/A3TGCN-VAR-PEM-Implementation.git"
REPO_BRANCH    = "main"

# Kaggle input folders (adjust to your dataset slugs)
DATA_DIR       = "/kaggle/input/largest-gba-2021"
ARTIFACT_DIR   = "/kaggle/input/ablation-artifacts"

## 1. Environment: Python version, TensorFlow 2.15 (last release with the TF1 API used by the model)

In [ ]:
import sys, subprocess
print("Python", sys.version)
assert sys.version_info[:2] <= (3, 11), "TensorFlow 2.15 needs Python <= 3.11"

# Installed into the environment; main.py runs in a fresh process, so no kernel restart is needed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tensorflow[and-cuda]==2.15.1", "numpy==1.26.4", "scipy==1.13.1",
                "h5py", "pandas<3", "joblib>=1.4"], check=True)

print(subprocess.run([sys.executable, "-c",
      "import tensorflow as tf, numpy as np; print('TF', tf.__version__, '| numpy', np.__version__); "
      "print('GPUs:', tf.config.list_physical_devices('GPU'))"],
      capture_output=True, text=True).stdout)

## 2. Code (fresh clone) and data / artifacts

In [ ]:
import os, glob, shutil
WORK = "/kaggle/working"
CODE = f"{WORK}/implementation"          # package name must be 'implementation'

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# Data: datasets/LargeST GBA/ -> read-only Kaggle input (no copy)
os.makedirs(f"{CODE}/datasets", exist_ok=True)
os.symlink(DATA_DIR, f"{CODE}/datasets/LargeST GBA")
for name in ["gba_his_raw_2021.h5", "gba_rn_adj.npy", "gba_meta.csv"]:
    assert os.path.exists(f"{CODE}/datasets/LargeST GBA/{name}"), f"missing {name} in {DATA_DIR}"

# Artifacts: graphs and the cached sensor filter (recomputed if absent, same result)
os.makedirs(f"{CODE}/outputs/graphs", exist_ok=True)
os.makedirs(f"{CODE}/outputs/sensor_filter", exist_ok=True)
if os.path.isdir(ARTIFACT_DIR):
    for path in glob.glob(f"{ARTIFACT_DIR}/**/*", recursive=True):
        if path.endswith((".npz", ".json")) and os.path.basename(path).startswith(("E1__", "E2__")):
            shutil.copy(path, f"{CODE}/outputs/graphs/")
        elif os.path.basename(path).startswith("sensor_filter_"):
            shutil.copy(path, f"{CODE}/outputs/sensor_filter/")
print("graphs:", os.listdir(f"{CODE}/outputs/graphs"))
print("sensor filter cache:", os.listdir(f"{CODE}/outputs/sensor_filter"))

## 3. Run

In [ ]:
cmd = [sys.executable, "-u", "-m", "implementation.main", "--experiment", EXPERIMENT]
if GRAPH_FILE:
    cmd += ["--graph", f"{CODE}/outputs/graphs/{GRAPH_FILE}"]
if EPOCHS is not None:
    cmd += ["--epochs", str(EPOCHS)]
if SELECTION_ONLY:
    cmd += ["--selection-only"]
if RUN_NAME:
    cmd += ["--run-name", RUN_NAME]
print(" ".join(cmd))

env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2", PYTHONUNBUFFERED="1")
with subprocess.Popen(cmd, cwd=WORK, env=env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        if not any(noise in line for noise in ("WARNING:tensorflow", "Instructions for updating", "non-resource variables")):
            print(line, end="")
assert process.returncode == 0, f"main.py exited with {process.returncode}"

## 4. Results (saved as notebook output)

In [ ]:
import json
run_dir = f"{CODE}/results/{RUN_NAME or EXPERIMENT}"
results = json.load(open(f"{run_dir}/results.json"))
summary = {k: results.get(k) for k in ["experiment", "best_epoch", "best_val_rmse_original",
                                         "training_seconds", "valid_sequences", "test_metrics"]}
print(json.dumps(summary, indent=2))

out = f"{WORK}/output_{RUN_NAME or EXPERIMENT}"
shutil.copytree(run_dir, out, dirs_exist_ok=True)
print("Saved to", out, "->", os.listdir(out))